# Inference Performance Comparison

In this notebook, we'll compare the inference performance of our baseline models versus the optimized models. We'll measure metrics like latency, throughput, and resource utilization.

## 1. Import Dependencies

In [ ]:
import os
import json
import time
import boto3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from datetime import datetime

# Import utility functions
from utils import plot_comparison

## 2. Load Workshop Settings

Load the workshop settings that were configured in the first notebook.

In [ ]:
# Load stored variables
%store -r S3_BUCKET
%store -r AWS_REGION
%store -r SAGEMAKER_ROLE_ARN

# Check if variables were successfully retrieved
if 'S3_BUCKET' in locals() and S3_BUCKET != "YOUR_BUCKET_NAME_HERE":
    print("Workshop settings loaded successfully:")
    print(f"S3 Bucket: {S3_BUCKET}")
    print(f"AWS Region: {AWS_REGION}")
    print(f"SageMaker Role ARN: {SAGEMAKER_ROLE_ARN}")
else:
    print("⚠️ Workshop settings not found or not configured.")
    print("Please run the first notebook (01_introduction_and_setup.ipynb) to configure settings.")
    
    # Set default values that user should update
    S3_BUCKET = "YOUR_BUCKET_NAME_HERE"  # Update this value
    AWS_REGION = "YOUR_REGION_HERE"      # Update this value
    SAGEMAKER_ROLE_ARN = "YOUR_ROLE_ARN_HERE"  # Update this value
    
    # Store the updated values
    %store S3_BUCKET
    %store AWS_REGION
    %store SAGEMAKER_ROLE_ARN

## 3. Load Performance Data

In [ ]:
# Load baseline metrics
try:
    with open('baseline_metrics.json', 'r') as f:
        baseline_metrics = json.load(f)
    print(f"Loaded baseline metrics for {len(baseline_metrics)} models")
except FileNotFoundError:
    print("baseline_metrics.json not found. Please run the baseline evaluation notebook.")
    baseline_metrics = {}

# Load quantized model metrics
try:
    with open('quantized_metrics.json', 'r') as f:
        quantized_metrics = json.load(f)
    print(f"Loaded quantized metrics for {len(quantized_metrics)} models")
except FileNotFoundError:
    print("quantized_metrics.json not found. Please run the quantization notebook.")
    quantized_metrics = {}

# Load pruned model metrics
try:
    with open('pruned_metrics.json', 'r') as f:
        pruned_metrics = json.load(f)
    print(f"Loaded pruned metrics for {len(pruned_metrics)} models")
except FileNotFoundError:
    print("pruned_metrics.json not found. Please run the pruning notebook.")
    pruned_metrics = {}

# Load distilled model metrics
try:
    with open('distilled_metrics.json', 'r') as f:
        distilled_metrics = json.load(f)
    print(f"Loaded distilled metrics for {len(distilled_metrics)} models")
except FileNotFoundError:
    print("distilled_metrics.json not found. Please run the knowledge distillation notebook.")
    distilled_metrics = {}

## 4. Create Performance Comparison DataFrame

In [ ]:
# Create a DataFrame for performance comparison
performance_data = []

# Process baseline metrics
for model_key, metrics in baseline_metrics.items():
    performance_data.append({
        "Model": metrics["model_name"],
        "Optimization": "Baseline",
        "Size (MB)": metrics["model_size"],
        "Inference Time (ms)": metrics["inference_time"],

        "Parameters": metrics["num_parameters"]
    })

# Process quantized metrics
for model_key, metrics in quantized_metrics.items():
    if model_key in baseline_metrics:  # Only include models that have baseline metrics
        performance_data.append({
            "Model": metrics["model_name"],
            "Optimization": "Quantized",
            "Size (MB)": metrics["model_size"],
            "Inference Time (ms)": metrics["inference_time"],

            "Parameters": metrics["num_parameters"]
        })

# Process pruned metrics
for model_key, metrics in pruned_metrics.items():
    if model_key in baseline_metrics:  # Only include models that have baseline metrics
        performance_data.append({
            "Model": metrics["model_name"],
            "Optimization": "Pruned",
            "Size (MB)": metrics["model_size"],
            "Inference Time (ms)": metrics["inference_time"],

            "Parameters": metrics["num_parameters"]
        })

# Process distilled metrics
for model_key, metrics in distilled_metrics.items():
    # For distilled models, we use the teacher model as the baseline
    teacher_key = metrics.get("teacher_model_key")
    if teacher_key and teacher_key in baseline_metrics:
        performance_data.append({
            "Model": metrics["model_name"],
            "Optimization": "Distilled",
            "Size (MB)": metrics["model_size"],
            "Inference Time (ms)": metrics["inference_time"],

            "Parameters": metrics["num_parameters"]
        })

# Create DataFrame
performance_df = pd.DataFrame(performance_data)

# Display the DataFrame
performance_df

## 5. Calculate Improvement Percentages

In [ ]:
# Create a pivot table to compare baseline vs optimized metrics
size_pivot = performance_df.pivot_table(index="Model", columns="Optimization", values="Size (MB)")
time_pivot = performance_df.pivot_table(index="Model", columns="Optimization", values="Inference Time (ms)")
memory_pivot = performance_df.pivot_table(index="Model", columns="Optimization", values="Memory Usage (MB)")
params_pivot = performance_df.pivot_table(index="Model", columns="Optimization", values="Parameters")

# Calculate improvement percentages
improvement_data = []

for model in size_pivot.index:
    baseline_size = size_pivot.loc[model, "Baseline"]
    baseline_time = time_pivot.loc[model, "Baseline"]
    baseline_memory = memory_pivot.loc[model, "Baseline"]
    baseline_params = params_pivot.loc[model, "Baseline"]
    
    for opt in ["Quantized", "Pruned", "Distilled"]:
        if opt in size_pivot.columns and not pd.isna(size_pivot.loc[model, opt]):
            opt_size = size_pivot.loc[model, opt]
            opt_time = time_pivot.loc[model, opt]
            opt_memory = memory_pivot.loc[model, opt]
            opt_params = params_pivot.loc[model, opt]
            
            size_reduction = (baseline_size - opt_size) / baseline_size * 100
            time_reduction = (baseline_time - opt_time) / baseline_time * 100
            params_reduction = (baseline_params - opt_params) / baseline_params * 100
            
            improvement_data.append({
                "Model": model,
                "Optimization": opt,
                "Size Reduction (%)": size_reduction,
                "Inference Time Reduction (%)": time_reduction,
                "Memory Usage Reduction (%)": memory_reduction,
                "Parameters Reduction (%)": params_reduction
            })

# Create DataFrame
improvement_df = pd.DataFrame(improvement_data)

# Display the DataFrame
improvement_df

## 6. Display Performance Comparison

In [ ]:
# Display a table with all metrics
performance_df.sort_values(["Model", "Optimization"])

## 7. Summary of Findings

In [ ]:
# Calculate average improvements by optimization technique
avg_improvements = improvement_df.groupby("Optimization").mean()

# Display average improvements
avg_improvements

## 8. Next Steps

Now that we've compared the performance of our baseline and optimized models, we'll analyze the cost implications in the next notebook.